**NOTE THAT THE FOLLOWING CODE WILL NOT WORK OUTSIDE OF THE WIKIMEDIA PAWS ENVIROMENT!**  
Go to https://paws.wmcloud.org, log into a Wikimedia account, and run the code within a Python3 notebook.

### **Library & Package Imports**

In [ ]:
import os
import pymysql
import json

### **Hard-Coded Data**

##### **Wikipedia URL title for each article edition**

In [ ]:
article_titles = {
    "John_F_Kennedy" : [ ("en", "John_F._Kennedy"),
                        ("ru", "Кеннеди,_Джон_Фицджералд"), 
                        ("es", "John_F._Kennedy"),
                        ("vi", "John_F._Kennedy") ],
    "Queen_Elizabeth_II" : [ ("en", "Elizabeth_II"),
                            ("hi", "एलिज़ाबेथ_द्वितीय"), 
                            ("ga", "Eilís_II_na_Ríochta_Aontaithe"),
                            ("af", "Elizabeth_II_van_die_Verenigde_Koninkryk") ],
    "Saddam_Hussein" : [ ("en", "Saddam_Hussein"),
                       ("ar", "صدام_حسين"),
                       ("fa", "صدام_حسین") ],
    "Mikhail_Gorbachev" : [ ("en", "Mikhail_Gorbachev"),
                           ("ru", "Горбачёв,_Михаил_Сергеевич"),
                           ("zh", "米哈伊尔·戈尔巴乔夫"),
                           ("de", "Michail_Sergejewitsch_Gorbatschow"),
                           ("be", "Міхаіл_Сяргеевіч_Гарбачоў") ],
    "Che_Guevara" : [ ("en", "Che_Guevara"),
                    ("es", "Che_Guevara"),
                    ("fr", "Che_Guevara"),
                    ("sw", "Che_Guevara") ],
    "Mother_Teresa" : [ ("bn", "মাদার_টেরিজা"),
                    ("sq", "Nënë_Tereza"),
                    ("ur", "مدر_ٹریسا"),
                    ("en", "Mother_Teresa") ],
    "Pope_John_Paul_II" : [ ("it", "Papa_Giovanni_Paolo_II"),
                           ("ar", "يوحنا_بولس_الثاني"),
                           ("pl", "Jan_Paweł_II"),
                           ("en", "Pope_John_Paul_II")],
    "Jean-Paul_Sartre" : [ ("fr", "Jean-Paul_Sartre"),
                         ("ru", "Сартр,_Жан-Поль"),
                         ("en", "Jean-Paul_Sartre") ],
    "Bob_Marley" : [ ("jam", "Bab_Maali"),
                    ("am", "ቦብ_ማርሊ"),
                    ("ja", "ボブ・マーリー"),
                    ("en", "Bob_Marley") ],
    "Umm_Kulthum" : [ ("arz", "ام_كلثوم"),
                    ("he", "אום_כולתום"),
                    ("ar", "أم_كلثوم_(مطربة)"),
                    ("en", "Umm_Kulthum") ],
    "Sirimavo_Bandaranaike" : [ ("si", "සිරිමාවෝ_බණ්ඩාරනායක"),
                              ("ta", "சிறிமாவோ_பண்டாரநாயக்கா"),
                              ("en", "Sirimavo_Bandaranaike") ],
    "Nelson_Mandela" : [ ("af", "Nelson_Mandela"),
                       ("zu", "Nelson_Mandela"),
                       ("ru", "Мандела,_Нельсон"),
                       ("en", "Nelson_Mandela")]
}

### **File Reading & Writing**

##### **Saving the (extracted) edit data to a JSON file**

In [ ]:
def save_data_to_file (data : list, file_path : str):
    """
    This function takes Wikipedia edit data, and saves it as a JSON file.

    :data: The edit data as a list of tuples, each in the format (edit timestamp, editor IP address, editor ID).
    :file_path: The file path where the data should be saved to.
    If the file already exists, you will have the option to (not) overwrite it.
    """
    if len(file_path) <= 5 or file_path[-5:] != ".json":
        # If file_path does not end with a JSON file
        print(f"'{file_path}' is not the path of a valid JSON file.")
    elif data == None:
        print("There is no edit data to be saved.")
    elif isinstance(data, list) == False:
        # Only save the data if it is a list
        print("The edit data could not be saved as it is invalid.")
    else:
        try:
            json_str = json.dumps(data)
            with open(file_path, "w") as file:
                file.write(json_str)
            print(f"The edit data has been saved to '{file_path}'.")
        except:
            print("The edit data could not be saved due to an error.")

##### **Saving edit data in bulk**

In [ ]:
def bulk_save_data (data_list : list, file_path_list : list):
    """
    This function saves the edit data of multiple Wikipedia articles to seperate JSON files.
    
    :data_list: A list of lists containing (what is assumed to be) each article's edit data.
    :file_path_list: A list of file paths denoting where each article's data should be saved to.
    If any file path points to an existing file, you will have the option to (not) overwrite it.
    """
    if len(data_list) > len(file_path_list):
        # If the list of file paths is not long enough to be paired with the lists of edit data
        print("Error - At least one file path must be provided for each article's edit data.")
    else:
        for data, file_path in zip(data_list, file_path_list):
            # Only save the data if it is a list
            if isinstance(data, list) == True:
                save_data_to_file(data, file_path)

### **Edit Data Extraction**

##### **Extracting the edit data (from the Wiki Replicas databases)**

In [ ]:
def extract_edit_data (edition_code : str, article_title : str, retry : bool):
    """
    NOTE: THIS FUNCTION WILL NOT WORK OUTSIDE OF PAWS (Wikimedia’s online Jupyter notebook environment).
    Go to https://paws.wmcloud.org, log into a Wikimedia account, and run this function within a Python3 notebook.
    
    This function retrieves data on every edit made (by an anonymous editor) to the specified Wikipedia article.
    
    :edition_code: The code for the target Wikipedia language edition.
    :article_title: The title of the target Wikipedia article (in the respective language edition).
    :retry: If set to True, and the edit data could not be retrived, then a different method will be used to extract it.
    
    Returns a list of edit data (edit timestamp, editor IP address, editor ID) 
    for every anonymous edit made to the article, or returns None if the data could not be retrieved at all.
    """
    data = []
    try:
        conn = pymysql.connect(
            host = edition_code + "wiki.web.db.svc.wikimedia.cloud",
            database = edition_code + "wiki_p",
            charset = "utf8mb4",
            read_default_file = "~/.my.cnf",
        )

        with conn.cursor() as cur:
            cur.execute(f""" SELECT i.ipc_rev_timestamp, i.ipc_hex, r.rev_actor 
            FROM ip_changes AS i, revision AS r, page AS p 
            WHERE p.page_title = '{article_title}' AND i.ipc_rev_id = r.rev_id AND r.rev_page = p.page_id 
            ORDER BY i.ipc_rev_timestamp DESC;""")
            
            fetch_data = cur.fetchall() # Retrieves the data returned from the Wiki database replica query.
            # The data is originally a list of nested tuples, and the timestamps and IP addresses encoded as bytes rather than integers.
            
            fetch_data = list(fetch_data) # Changes the data from a tuple to a list
            fetch_data = map(lambda tup : ( int(str(tup[0])[2:-1]) , str(tup[1])[2:-1] , tup[2] ) , fetch_data) # Corrects the timestamp and IP address types.
            fetch_data = list(fetch_data) # Changes the data from a map object to a list
            data.extend(fetch_data)
                
        print(f"The edit data for the article '{article_title}' ({edition_code}) was successfully retrieved.")
        conn.close()
        return data
            
    except Exception as e:
        print(f"Error: {e}")
        if retry == True:
            print("Retrying edit data extraction...")
            return special_extract_edit_data(edition_code, article_title)
        else:
            return None


def special_extract_edit_data (edition_code : str, article_title : str):
    """
    NOTE: THIS FUNCTION WILL NOT WORK OUTSIDE OF PAWS (Wikimedia’s online Jupyter notebook environment).
    Go to https://paws.wmcloud.org, log into your Wikimedia account, and run this function within a Python3 notebook.
    
    This function retrieves data on every edit made (by an anonymous editor) to the specified Wikipedia article.
    It uses a more reliable (albeit slower) method for this compared to the extract_edit_data() function.
    
    :edition_code: The code for the target Wikipedia language edition.
    :article_title: The title of the target Wikipedia article (in the respective language edition).
    
    Returns a list of edit data (edit timestamp, editor IP address, editor ID) 
    for every anonymous edit made to the article, or returns None if the data could not be retrieved at all.
    """
    data = []
    timestamp = 20260000000000
    retry_count = 0
    
    while timestamp >= 20010000000000:
        # The timestamp variable is used to query for article edits made in a specific year.
        # This allows the edit data from each year (working backwards from 2026) to be aggregated.
        try:
            conn = pymysql.connect(
                host = edition_code + "wiki.web.db.svc.wikimedia.cloud",
                database = edition_code + "wiki_p",
                charset = "utf8mb4",
                read_default_file = "~/.my.cnf",
                connect_timeout = 10000
            )
    
            with conn.cursor() as cur:
                cur.execute(f""" SELECT i.ipc_rev_timestamp, i.ipc_hex, r.rev_actor 
                FROM ip_changes AS i, revision AS r, page AS p 
                WHERE i.ipc_rev_timestamp BETWEEN {timestamp} AND {timestamp + 10000000000} 
                AND p.page_title = '{article_title}' AND i.ipc_rev_id = r.rev_id AND r.rev_page = p.page_id 
                ORDER BY i.ipc_rev_timestamp DESC;""")
                
                fetch_data = cur.fetchall() # Retrieves the data returned from the Wiki database replica query.
                # The data is originally a list of nested tuples, and the timestamps and IP addresses encoded as bytes rather than integers.
                
                fetch_data = list(fetch_data) # Changes the data from a tuple to a list
                fetch_data = map(lambda tup : ( int(str(tup[0])[2:-1]) , str(tup[1])[2:-1] , tup[2] ) , fetch_data) # Corrects the timestamp and IP address types.
                fetch_data = list(fetch_data) # Changes the data from a map object to a list
                data.extend(fetch_data)

                if timestamp == 20260000000000:
                    print(f"Data extracted for the year", end=' ')
                print(f"{int(timestamp/10000000000)}...", end=' ')
                timestamp -= 10000000000
                conn.close()
             
        except Exception as e:
            # The most common exception occurs when the connection to the SQL database is terminated (timed out)
            print("(error)...", end=' ')
            if retry_count < 10:
                # Allow 10 retries before returning the data that has been collected so far
                retry_count += 1
            elif data == []:
                # If none of the edit data could be extracted
                print(f"Error: {e}")
                print(f"\nThe edit data for the article '{article_title}' ({edition_code}) could not be retrieved.")
                return None
            else:
                # If some but not all of the edit data could be extracted
                print(f"Error: {e}")
                print(f"\nThe edit data for the article '{article_title}' ({edition_code}) could only be retrieved for the year {int(timestamp/10000000000) + 1} and onwards.")
                return data
            
    print(f"\nThe edit data for the article '{article_title}' ({edition_code}) was successfully retrieved.")
    return data

##### **Extracting edit data in bulk**

In [ ]:
def bulk_extract_edit_data (article_titles, retry : bool):
    """
    NOTE: THIS FUNCTION WILL NOT WORK OUTSIDE OF PAWS (Wikimedia’s online Jupyter notebook environment).
    Go to https://paws.wmcloud.org, log into your Wikimedia account, and run this function within a Python3 notebook.
    
    This function retrieves the edit data of multiple Wikipedia articles / article editions.
    
    :article_titles: A list of tuples, each in the format (language edition code, article title).
    Note that the same article will likely have a different title in different language editions.
    :retry: If set to True, and the edit data could not be retrived, then a different method will be used to extract it.
    
    Returns a dictionary in the format of { title & edition code : edit data (as a list) }.
    If an article's edit data cannot be retrieved, its dictionary value will be None.
    """
    data_dict = dict()

    for article in article_titles:
        data = extract_edit_data(article[0], article[1], retry)
        data_dict[article[1] + " " + article[0]] = data
            
    return data_dict

### **Code Execution Area**

##### **Extracting and saving the edit data for a single article edition**

In [ ]:
data = extract_edit_data("en", "John_F._Kennedy", True)
save_data_to_file(data, "Edit_Data_English.json")

##### **Extracting and saving the edit data for all editions of an article**

In [ ]:
### John F. Kennedy's article
bulk_data = bulk_extract_edit_data(article_titles["John_F_Kennedy"], True)

file_path_list = ["Edit_Data_English.json", "Edit_Data_Russian.json", 
                  "Edit_Data_Spanish.json", "Edit_Data_Vietnamese.json"]

bulk_save_data( list(bulk_data.values()), file_path_list )